In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import cross_val_score
from sklearn.ensemble import GradientBoostingRegressor

In [2]:
# TWEAK THESE LATER #

# Lower k = More aggressive = smaller spread

k_map = {
    1: 1.00,
    2: 1.00,
    3: 1.25,
    4: 1.50,
    5: 1.50,
    6: 2.00,
    7: 1.75,
    8: 2.00,
    9: 2.50
}

In [3]:
# BEST MODELS TO USE FOR EACH STOCK #

best_model_map = {
    1: "Ridge",
    2: "Ridge",
    3: "Ridge",
    4: "GradientBoosting",
    5: "Ridge",
    6: "Ridge",
    7: "GradientBoosting",   # basically a tie, but GB won very slightly
    8: "Ridge",
    9: "GradientBoosting"
}

In [41]:
# VERSION 2 WITH THE UPDATED MODELS #

best_model_map_v2 = {
    1: "LinearRegression",
    2: "LinearRegression",
    3: "Ridge",
    4: "XGBoost",
    5: "Ridge",
    6: "Ridge",
    7: "XGBoost",
    8: "Ridge",
    9: "LightGBM"
}

In [4]:
def evaluate_models(train_csv_path, random_state=42):
    df = pd.read_csv(train_csv_path)

    X = df.drop(columns=["target"])
    y = df["target"]

    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=random_state
    )

    models = {
        "LinearRegression": Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler", StandardScaler()),
            ("model", LinearRegression())
        ]),

        "Ridge": Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=1.0))
        ]),

        "RandomForest": Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("model", RandomForestRegressor(
                n_estimators=200,
                max_depth=None,
                random_state=random_state,
                n_jobs=-1
            ))
        ])
    }

    results = []
    fitted_models = {}

    for name, model in models.items():
        model.fit(X_train, y_train)
        preds = model.predict(X_val)

        mae = mean_absolute_error(y_val, preds)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        r2 = r2_score(y_val, preds)

        results.append({
            "model": name,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2
        })

        fitted_models[name] = model

    results_df = pd.DataFrame(results).sort_values("RMSE")
    return results_df, fitted_models

In [5]:
results_df, fitted_models = evaluate_models("hackathon_data/stock_1_train.csv")
results_df

,model,MAE,RMSE,R2
0,LinearRegression,3.833588,4.813408,0.98443
1,Ridge,3.833648,4.813458,0.98443
2,RandomForest,4.390848,5.678696,0.97833


In [6]:
def fit_ridge_and_predict(train_csv_path, test_csv_path):
    train_df = pd.read_csv(train_csv_path)
    test_df = pd.read_csv(test_csv_path)

    X_train = train_df.drop(columns=["target"])
    y_train = train_df["target"]

    ridge_model = Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0))
    ])

    ridge_model.fit(X_train, y_train)
    predictions = ridge_model.predict(test_df)

    return predictions

In [7]:
def predict_test_file(train_csv_path, test_csv_path, chosen_model_name=None, random_state=42):
    # Evaluate candidate models first
    results_df, fitted_models = evaluate_models(train_csv_path, random_state=random_state)
    
    if chosen_model_name is None:
        chosen_model_name = results_df.iloc[0]["model"]  # best RMSE
    
    # Reload full training data
    train_df = pd.read_csv(train_csv_path)
    X_train_full = train_df.drop(columns=["target"])
    y_train_full = train_df["target"]
    
    # Get chosen model and retrain on ALL training data
    model = fitted_models[chosen_model_name]
    model.fit(X_train_full, y_train_full)
    
    # Load test data
    test_df = pd.read_csv(test_csv_path)
    
    # Predict
    preds = model.predict(test_df)
    
    output = test_df.copy()
    output["predicted_target"] = preds
    
    return results_df, chosen_model_name, output

In [8]:
results_df, best_model_name, predictions_df = predict_test_file(
    "hackathon_data/stock_1_train.csv",
    "hackathon_data/stock_1_test.csv"
)

print("Best model:", best_model_name)
print(results_df)
predictions_df.head()

Best model: LinearRegression
              model       MAE      RMSE       R2
0  LinearRegression  3.833588  4.813408  0.98443
1             Ridge  3.833648  4.813458  0.98443
2      RandomForest  4.390848  5.678696  0.97833


,col_0,col_1,col_2,col_3,col_4,predicted_target
0,-1.25546,0.610359,0.26401,-0.803538,0.819936,273.929337


In [9]:
predictions_df.to_csv("stock_1_predictions.csv", index=False)

In [10]:
all_results = {}

for i in range(1, 10):
    train_path = f"hackathon_data/stock_{i}_train.csv"
    test_path = f"hackathon_data/stock_{i}_test.csv"
    
    results_df, best_model_name, predictions_df = predict_test_file(train_path, test_path)
    
    all_results[i] = {
        "results": results_df,
        "best_model": best_model_name,
        "predictions": predictions_df
    }
    
    print(f"\nStock {i}")
    print("Best model:", best_model_name)
    print(results_df)


Stock 1
Best model: LinearRegression
              model       MAE      RMSE       R2
0  LinearRegression  3.833588  4.813408  0.98443
1             Ridge  3.833648  4.813458  0.98443
2      RandomForest  4.390848  5.678696  0.97833

Stock 2
Best model: Ridge
              model        MAE       RMSE        R2
1             Ridge   7.606596   9.831272  0.959606
0  LinearRegression   7.610278   9.831789  0.959602
2      RandomForest  17.363345  21.844847  0.800568

Stock 3
Best model: LinearRegression
              model        MAE       RMSE        R2
0  LinearRegression  19.132499  24.409742  0.872108
1             Ridge  20.458676  25.623765  0.859070
2      RandomForest  27.175391  33.262447  0.762521

Stock 4
Best model: RandomForest
              model        MAE       RMSE        R2
2      RandomForest  12.581804  25.036638  0.172066
1             Ridge  13.648821  25.763352  0.123305
0  LinearRegression  13.648890  25.763394  0.123302

Stock 5
Best model: Ridge
              mo

In [11]:
def make_quotes(predicted_fair_value, rmse, k):
    half_spread = k * rmse
    bid = round(predicted_fair_value - half_spread, 2)
    ask = round(predicted_fair_value + half_spread, 2)
    spread_width = round(ask - bid, 2)
    return bid, ask, spread_width

In [12]:
all_quotes = {}

for i in range(1, 10):
    train_path = f"hackathon_data/stock_{i}_train.csv"
    test_path = f"hackathon_data/stock_{i}_test.csv"

    # Get validation results
    results_df, _ = evaluate_models(train_path)

    # Get Ridge RMSE as uncertainty
    ridge_row = results_df[results_df["model"] == "Ridge"].iloc[0]
    ridge_rmse = ridge_row["RMSE"]

    # Predict fair values using Ridge trained on full training data
    fair_values = fit_ridge_and_predict(train_path, test_path)

    # Use stock-specific aggressiveness
    k = k_map[i]

    stock_quotes = []

    for j, fair_value in enumerate(fair_values):
        bid, ask, spread = make_quotes(fair_value, ridge_rmse, k=k)

        stock_quotes.append({
            "row": j,
            "predicted_fair_value": fair_value,
            "rmse_uncertainty": ridge_rmse,
            "k": k,
            "bid": bid,
            "ask": ask,
            "spread_width": spread
        })

    stock_quotes_df = pd.DataFrame(stock_quotes)
    all_quotes[i] = stock_quotes_df

    print(f"\nStock {i}")
    print(f"Ridge RMSE: {ridge_rmse:.4f}")
    print(f"k used: {k}")
    print(stock_quotes_df.head())


Stock 1
Ridge RMSE: 4.8135
k used: 1.0
   row  predicted_fair_value  rmse_uncertainty    k     bid     ask  \
0    0            273.927924          4.813458  1.0  269.11  278.74   

   spread_width  
0          9.63  

Stock 2
Ridge RMSE: 9.8313
k used: 1.0
   row  predicted_fair_value  rmse_uncertainty    k     bid     ask  \
0    0            220.911465          9.831272  1.0  211.08  230.74   

   spread_width  
0         19.66  

Stock 3
Ridge RMSE: 25.6238
k used: 1.25
   row  predicted_fair_value  rmse_uncertainty     k     bid     ask  \
0    0            257.823684         25.623765  1.25  225.79  289.85   

   spread_width  
0         64.06  

Stock 4
Ridge RMSE: 25.7634
k used: 1.5
   row  predicted_fair_value  rmse_uncertainty    k    bid     ask  \
0    0            248.843885         25.763352  1.5  210.2  287.49   

   spread_width  
0         77.29  

Stock 5
Ridge RMSE: 23.6749
k used: 1.5
   row  predicted_fair_value  rmse_uncertainty    k     bid     ask  \
0    0   

In [13]:
summary_rows = []

for i in range(1, 10):
    row = all_quotes[i].iloc[0]
    summary_rows.append({
        "stock": i,
        "predicted_fair_value": round(row["predicted_fair_value"], 4),
        "rmse_uncertainty": round(row["rmse_uncertainty"], 4),
        "k": row["k"],
        "bid": round(row["bid"], 4),
        "ask": round(row["ask"], 4),
        "spread_width": round(row["spread_width"], 4)
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

,stock,predicted_fair_value,rmse_uncertainty,k,bid,ask,spread_width
0,1,273.9279,4.8135,1.00,269.11,278.74,9.63
1,2,220.9115,9.8313,1.00,211.08,230.74,19.66
2,3,257.8237,25.6238,1.25,225.79,289.85,64.06
3,4,248.8439,25.7634,1.50,210.20,287.49,77.29
4,5,249.0281,23.6749,1.50,213.52,284.54,71.02
5,6,174.9202,70.7595,2.00,33.40,316.44,283.04
6,7,214.7137,16.1006,1.75,186.54,242.89,56.35
7,8,200.5979,25.9233,2.00,148.75,252.44,103.69
8,9,219.9991,45.0651,2.50,107.34,332.66,225.32


In [14]:
def get_model(model_name):
    if model_name == "Ridge":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=1.0))
        ])
    
    elif model_name == "GradientBoosting":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("model", GradientBoostingRegressor(
                n_estimators=200,
                learning_rate=0.05,
                max_depth=3,
                random_state=42
            ))
        ])
    
    else:
        raise ValueError(f"Unknown model name: {model_name}")

In [15]:
def compare_models_cv(train_csv_path, cv=5):
    df = pd.read_csv(train_csv_path)

    X = df.drop(columns=["target"])
    y = df["target"]

    models = {
        "Ridge": Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=1.0))
        ]),

        "GradientBoosting": Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("model", GradientBoostingRegressor(
                n_estimators=200,
                learning_rate=0.05,
                max_depth=3,
                random_state=42
            ))
        ])
    }

    rows = []

    for model_name, model in models.items():
        rmse_scores = -cross_val_score(
            model,
            X,
            y,
            cv=cv,
            scoring="neg_root_mean_squared_error",
            n_jobs=-1
        )

        mae_scores = -cross_val_score(
            model,
            X,
            y,
            cv=cv,
            scoring="neg_mean_absolute_error",
            n_jobs=-1
        )

        r2_scores = cross_val_score(
            model,
            X,
            y,
            cv=cv,
            scoring="r2",
            n_jobs=-1
        )

        rows.append({
            "model": model_name,
            "CV_RMSE_mean": rmse_scores.mean(),
            "CV_RMSE_std": rmse_scores.std(),
            "CV_MAE_mean": mae_scores.mean(),
            "CV_MAE_std": mae_scores.std(),
            "CV_R2_mean": r2_scores.mean(),
            "CV_R2_std": r2_scores.std()
        })

    results_df = pd.DataFrame(rows).sort_values("CV_RMSE_mean")
    return results_df

In [16]:
compare_models_cv("hackathon_data/stock_1_train.csv")

,model,CV_RMSE_mean,CV_RMSE_std,CV_MAE_mean,CV_MAE_std,CV_R2_mean,CV_R2_std
0,Ridge,4.867444,0.030134,3.88224,0.020972,0.984457,0.000203
1,GradientBoosting,5.281895,0.047347,4.19852,0.034913,0.981695,0.000415


In [17]:
cv_results_all = {}

for i in range(1, 10):
    train_path = f"hackathon_data/stock_{i}_train.csv"
    results_df = compare_models_cv(train_path, cv=5)
    cv_results_all[i] = results_df

    print(f"\nStock {i}")
    print(results_df)


Stock 1
              model  CV_RMSE_mean  CV_RMSE_std  CV_MAE_mean  CV_MAE_std  \
0             Ridge      4.867444     0.030134      3.88224    0.020972   
1  GradientBoosting      5.281895     0.047347      4.19852    0.034913   

   CV_R2_mean  CV_R2_std  
0    0.984457   0.000203  
1    0.981695   0.000415  

Stock 2
              model  CV_RMSE_mean  CV_RMSE_std  CV_MAE_mean  CV_MAE_std  \
0             Ridge      9.694097     0.446819     7.709752    0.379514   
1  GradientBoosting     17.514501     0.429420    13.605256    0.213989   

   CV_R2_mean  CV_R2_std  
0    0.959835   0.004376  
1    0.869358   0.005366  

Stock 3
              model  CV_RMSE_mean  CV_RMSE_std  CV_MAE_mean  CV_MAE_std  \
0             Ridge     25.922219     9.432560    20.323423    6.666537   
1  GradientBoosting     43.966398    14.522501    37.389360   12.791247   

   CV_R2_mean  CV_R2_std  
0    0.763706   0.253149  
1    0.297936   0.521705  

Stock 4
              model  CV_RMSE_mean  CV_RMSE_

In [18]:
summary_rows = []

for i in range(1, 10):
    results_df = cv_results_all[i]
    best_row = results_df.iloc[0]
    ridge_row = results_df[results_df["model"] == "Ridge"].iloc[0]
    gb_row = results_df[results_df["model"] == "GradientBoosting"].iloc[0]

    summary_rows.append({
        "stock": i,
        "best_model": best_row["model"],
        "best_CV_RMSE": round(best_row["CV_RMSE_mean"], 4),
        "ridge_CV_RMSE": round(ridge_row["CV_RMSE_mean"], 4),
        "gb_CV_RMSE": round(gb_row["CV_RMSE_mean"], 4),
        "gb_minus_ridge_rmse": round(gb_row["CV_RMSE_mean"] - ridge_row["CV_RMSE_mean"], 4)
    })

cv_summary_df = pd.DataFrame(summary_rows)
cv_summary_df

,stock,best_model,best_CV_RMSE,ridge_CV_RMSE,gb_CV_RMSE,gb_minus_ridge_rmse
0,1,Ridge,4.8674,4.8674,5.2819,0.4145
1,2,Ridge,9.6941,9.6941,17.5145,7.8204
2,3,Ridge,25.9222,25.9222,43.9664,18.0442
3,4,GradientBoosting,24.7461,25.7868,24.7461,-1.0407
4,5,Ridge,27.9683,27.9683,29.8082,1.8399
5,6,Ridge,56.0904,56.0904,62.8534,6.7631
6,7,GradientBoosting,15.5156,15.5448,15.5156,-0.0292
7,8,Ridge,26.1218,26.1218,26.5728,0.4510
8,9,GradientBoosting,52.9891,61.9464,52.9891,-8.9574


In [19]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def old_split_ridge_score(train_csv_path, random_state=42):
    df = pd.read_csv(train_csv_path)

    X = df.drop(columns=["target"])
    y = df["target"]

    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=random_state
    )

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0))
    ])

    model.fit(X_train, y_train)
    preds = model.predict(X_val)

    rmse = np.sqrt(mean_squared_error(y_val, preds))
    mae = mean_absolute_error(y_val, preds)
    r2 = r2_score(y_val, preds)

    return {
        "old_RMSE": rmse,
        "old_MAE": mae,
        "old_R2": r2
    }

In [20]:
improvement_rows = []

for i in range(1, 10):
    train_path = f"hackathon_data/stock_{i}_train.csv"

    old_scores = old_split_ridge_score(train_path)
    new_results = compare_models_cv(train_path, cv=5)
    best_new = new_results.iloc[0]

    improvement_rows.append({
        "stock": i,
        "old_system_model": "Ridge_split",
        "old_RMSE": round(old_scores["old_RMSE"], 4),
        "new_system_model": best_new["model"],
        "new_CV_RMSE": round(best_new["CV_RMSE_mean"], 4),
        "change_in_rmse": round(best_new["CV_RMSE_mean"] - old_scores["old_RMSE"], 4)
    })

improvement_df = pd.DataFrame(improvement_rows)
improvement_df

,stock,old_system_model,old_RMSE,new_system_model,new_CV_RMSE,change_in_rmse
0,1,Ridge_split,4.8135,Ridge,4.8674,0.0540
1,2,Ridge_split,9.8313,Ridge,9.6941,-0.1372
2,3,Ridge_split,25.6238,Ridge,25.9222,0.2985
3,4,Ridge_split,25.7634,GradientBoosting,24.7461,-1.0172
4,5,Ridge_split,23.6749,Ridge,27.9683,4.2934
5,6,Ridge_split,70.7595,Ridge,56.0904,-14.6692
6,7,Ridge_split,16.1006,GradientBoosting,15.5156,-0.5850
7,8,Ridge_split,25.9233,Ridge,26.1218,0.1985
8,9,Ridge_split,45.0651,GradientBoosting,52.9891,7.9240


In [21]:
def fit_best_model_and_predict(train_csv_path, test_csv_path, best_model_name):
    train_df = pd.read_csv(train_csv_path)
    test_df = pd.read_csv(test_csv_path)

    X_train = train_df.drop(columns=["target"])
    y_train = train_df["target"]

    if best_model_name == "Ridge":
        model = Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=1.0))
        ])
    elif best_model_name == "GradientBoosting":
        model = Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("model", GradientBoostingRegressor(
                n_estimators=200,
                learning_rate=0.05,
                max_depth=3,
                random_state=42
            ))
        ])
    else:
        raise ValueError("Unknown model name")

    model.fit(X_train, y_train)
    predictions = model.predict(test_df)

    return predictions

In [22]:
cv_summary_df

,stock,best_model,best_CV_RMSE,ridge_CV_RMSE,gb_CV_RMSE,gb_minus_ridge_rmse
0,1,Ridge,4.8674,4.8674,5.2819,0.4145
1,2,Ridge,9.6941,9.6941,17.5145,7.8204
2,3,Ridge,25.9222,25.9222,43.9664,18.0442
3,4,GradientBoosting,24.7461,25.7868,24.7461,-1.0407
4,5,Ridge,27.9683,27.9683,29.8082,1.8399
5,6,Ridge,56.0904,56.0904,62.8534,6.7631
6,7,GradientBoosting,15.5156,15.5448,15.5156,-0.0292
7,8,Ridge,26.1218,26.1218,26.5728,0.4510
8,9,GradientBoosting,52.9891,61.9464,52.9891,-8.9574


In [23]:
improvement_df

,stock,old_system_model,old_RMSE,new_system_model,new_CV_RMSE,change_in_rmse
0,1,Ridge_split,4.8135,Ridge,4.8674,0.0540
1,2,Ridge_split,9.8313,Ridge,9.6941,-0.1372
2,3,Ridge_split,25.6238,Ridge,25.9222,0.2985
3,4,Ridge_split,25.7634,GradientBoosting,24.7461,-1.0172
4,5,Ridge_split,23.6749,Ridge,27.9683,4.2934
5,6,Ridge_split,70.7595,Ridge,56.0904,-14.6692
6,7,Ridge_split,16.1006,GradientBoosting,15.5156,-0.5850
7,8,Ridge_split,25.9233,Ridge,26.1218,0.1985
8,9,Ridge_split,45.0651,GradientBoosting,52.9891,7.9240


In [24]:
def get_cv_rmse(train_csv_path, model_name, cv=5):
    df = pd.read_csv(train_csv_path)
    
    X = df.drop(columns=["target"])
    y = df["target"]
    
    model = get_model(model_name)
    
    rmse_scores = -cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    )
    
    return rmse_scores.mean(), rmse_scores.std()

In [25]:
def fit_model_and_predict(train_csv_path, test_csv_path, model_name):
    train_df = pd.read_csv(train_csv_path)
    test_df = pd.read_csv(test_csv_path)
    
    X_train = train_df.drop(columns=["target"])
    y_train = train_df["target"]
    
    model = get_model(model_name)
    model.fit(X_train, y_train)
    
    predictions = model.predict(test_df)
    return predictions

In [26]:
def make_quotes(predicted_fair_value, cv_rmse, k=1.0):
    half_spread = k * cv_rmse
    bid = round(predicted_fair_value - half_spread, 2)
    ask = round(predicted_fair_value + half_spread, 2)
    spread_width = round(ask - bid, 2)
    return bid, ask, spread_width

In [27]:
all_quotes_cv = {}
summary_rows = []

for i in range(1, 10):
    train_path = f"hackathon_data/stock_{i}_train.csv"
    test_path = f"hackathon_data/stock_{i}_test.csv"
    
    # Choose best model for this stock
    model_name = best_model_map[i]
    
    # Get CV RMSE uncertainty
    cv_rmse_mean, cv_rmse_std = get_cv_rmse(train_path, model_name, cv=5)
    
    # Predict fair value(s) using full training data
    fair_values = fit_model_and_predict(train_path, test_path, model_name)
    
    # Use stock-specific aggressiveness
    k = k_map[i]
    
    stock_quotes = []
    
    for j, fair_value in enumerate(fair_values):
        bid, ask, spread = make_quotes(fair_value, cv_rmse_mean, k=k)
        
        stock_quotes.append({
            "row": j,
            "model_used": model_name,
            "predicted_fair_value": round(fair_value, 4),
            "cv_rmse_uncertainty": round(cv_rmse_mean, 4),
            "cv_rmse_std": round(cv_rmse_std, 4),
            "k": k,
            "bid": bid,
            "ask": ask,
            "spread_width": spread
        })
    
    stock_quotes_df = pd.DataFrame(stock_quotes)
    all_quotes_cv[i] = stock_quotes_df
    
    first_row = stock_quotes_df.iloc[0]
    summary_rows.append({
        "stock": i,
        "model_used": first_row["model_used"],
        "predicted_fair_value": first_row["predicted_fair_value"],
        "cv_rmse_uncertainty": first_row["cv_rmse_uncertainty"],
        "cv_rmse_std": first_row["cv_rmse_std"],
        "k": first_row["k"],
        "bid": first_row["bid"],
        "ask": first_row["ask"],
        "spread_width": first_row["spread_width"]
    })

In [28]:
summary_cv_df = pd.DataFrame(summary_rows)
summary_cv_df

,stock,model_used,predicted_fair_value,cv_rmse_uncertainty,cv_rmse_std,k,bid,ask,spread_width
0,1,Ridge,273.9279,4.8674,0.0301,1.00,269.06,278.80,9.74
1,2,Ridge,220.9115,9.6941,0.4468,1.00,211.22,230.61,19.39
2,3,Ridge,257.8237,25.9222,9.4326,1.25,225.42,290.23,64.81
3,4,GradientBoosting,235.8573,24.7461,0.4684,1.50,198.74,272.98,74.24
4,5,Ridge,249.0281,27.9683,3.9255,1.50,207.08,290.98,83.90
5,6,Ridge,174.9202,56.0904,14.3739,2.00,62.74,287.10,224.36
6,7,GradientBoosting,213.8173,15.5156,0.2764,1.75,186.66,240.97,54.31
7,8,Ridge,200.5979,26.1218,0.5956,2.00,148.35,252.84,104.49
8,9,GradientBoosting,210.1559,52.9891,12.5792,2.50,77.68,342.63,264.95


p = predicted fair value price
k = aggressiveness factor (lower k = more aggressive)
CV RMSE = uncertainty based on cross validation

bid = p - k * CV RMSE
ask = p + k * CV RMSE

For each stock:

- predicted_fair_value = your model’s best estimate of the true value
- cv_rmse_uncertainty = your typical prediction error from cross-validation
- cv_rmse_std = how much that RMSE varied across the folds
- k = how cautious/aggressive you are being
- bid / ask = final quote
- spread_width = total width of your spread

In [29]:
def classify_stock(row):
    rmse = row["cv_rmse_uncertainty"]
    rmse_std = row["cv_rmse_std"]
    spread = row["spread_width"]

    # Risk label
    if rmse < 10:
        risk_label = "Low"
        action = "Aggressive candidate"
    elif rmse < 20:
        risk_label = "Medium"
        action = "Moderate"
    elif rmse < 35:
        risk_label = "High"
        action = "Defensive"
    else:
        risk_label = "Very High"
        action = "Avoid market making"

    # Extra caution if CV error varies a lot across folds
    if rmse_std > 8:
        action = action + " (unstable CV)"

    return pd.Series([risk_label, action])

summary_cv_df[["risk_label", "action_recommendation"]] = summary_cv_df.apply(
    classify_stock, axis=1
)

summary_cv_df

,stock,model_used,predicted_fair_value,cv_rmse_uncertainty,cv_rmse_std,k,bid,ask,spread_width,risk_label,action_recommendation
0,1,Ridge,273.9279,4.8674,0.0301,1.00,269.06,278.80,9.74,Low,Aggressive candidate
1,2,Ridge,220.9115,9.6941,0.4468,1.00,211.22,230.61,19.39,Low,Aggressive candidate
2,3,Ridge,257.8237,25.9222,9.4326,1.25,225.42,290.23,64.81,High,Defensive (unstable CV)
3,4,GradientBoosting,235.8573,24.7461,0.4684,1.50,198.74,272.98,74.24,High,Defensive
4,5,Ridge,249.0281,27.9683,3.9255,1.50,207.08,290.98,83.90,High,Defensive
5,6,Ridge,174.9202,56.0904,14.3739,2.00,62.74,287.10,224.36,Very High,Avoid market making (unstable CV)
6,7,GradientBoosting,213.8173,15.5156,0.2764,1.75,186.66,240.97,54.31,Medium,Moderate
7,8,Ridge,200.5979,26.1218,0.5956,2.00,148.35,252.84,104.49,High,Defensive
8,9,GradientBoosting,210.1559,52.9891,12.5792,2.50,77.68,342.63,264.95,Very High,Avoid market making (unstable CV)


In [30]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor

def compare_many_models_cv(train_csv_path, cv=5):
    df = pd.read_csv(train_csv_path)

    X = df.drop(columns=["target"])
    y = df["target"]

    models = {
        "LinearRegression": Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler", StandardScaler()),
            ("model", LinearRegression())
        ]),

        "Ridge": Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=1.0))
        ]),

        "GradientBoosting": Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("model", GradientBoostingRegressor(
                n_estimators=200,
                learning_rate=0.05,
                max_depth=3,
                random_state=42
            ))
        ]),

        "NeuralNet_MLP": Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler", StandardScaler()),
            ("model", MLPRegressor(
                hidden_layer_sizes=(64, 32),
                activation="relu",
                solver="adam",
                alpha=0.0001,
                max_iter=1000,
                random_state=42
            ))
        ])
    }

    # Add XGBoost if available
    try:
        from xgboost import XGBRegressor
        models["XGBoost"] = Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("model", XGBRegressor(
                n_estimators=200,
                learning_rate=0.05,
                max_depth=4,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                objective="reg:squarederror"
            ))
        ])
    except ImportError:
        pass

    # Add LightGBM if available
    try:
        from lightgbm import LGBMRegressor
        models["LightGBM"] = Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("model", LGBMRegressor(
                n_estimators=200,
                learning_rate=0.05,
                max_depth=-1,
                random_state=42
            ))
        ])
    except ImportError:
        pass

    rows = []

    for model_name, model in models.items():
        rmse_scores = -cross_val_score(
            model,
            X,
            y,
            cv=cv,
            scoring="neg_root_mean_squared_error",
            n_jobs=-1
        )

        mae_scores = -cross_val_score(
            model,
            X,
            y,
            cv=cv,
            scoring="neg_mean_absolute_error",
            n_jobs=-1
        )

        r2_scores = cross_val_score(
            model,
            X,
            y,
            cv=cv,
            scoring="r2",
            n_jobs=-1
        )

        rows.append({
            "model": model_name,
            "CV_RMSE_mean": rmse_scores.mean(),
            "CV_RMSE_std": rmse_scores.std(),
            "CV_MAE_mean": mae_scores.mean(),
            "CV_R2_mean": r2_scores.mean()
        })

    return pd.DataFrame(rows).sort_values("CV_RMSE_mean")

In [31]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
import pandas as pd

def compare_core_models_cv(train_csv_path, cv=5):
    df = pd.read_csv(train_csv_path)

    X = df.drop(columns=["target"])
    y = df["target"]

    models = {
        "LinearRegression": Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler", StandardScaler()),
            ("model", LinearRegression())
        ]),

        "Ridge": Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=1.0))
        ]),

        "GradientBoosting": Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("model", GradientBoostingRegressor(
                n_estimators=100,
                learning_rate=0.05,
                max_depth=3,
                random_state=42
            ))
        ])
    }

    rows = []

    for model_name, model in models.items():
        rmse_scores = -cross_val_score(
            model,
            X,
            y,
            cv=cv,
            scoring="neg_root_mean_squared_error",
            n_jobs=-1
        )

        rows.append({
            "model": model_name,
            "CV_RMSE_mean": rmse_scores.mean(),
            "CV_RMSE_std": rmse_scores.std()
        })

    return pd.DataFrame(rows).sort_values("CV_RMSE_mean")

In [32]:
compare_core_models_cv("hackathon_data/stock_1_train.csv")

,model,CV_RMSE_mean,CV_RMSE_std
0,LinearRegression,4.867444,0.030132
1,Ridge,4.867444,0.030134
2,GradientBoosting,6.855602,0.075542


In [33]:
core_model_results = {}

for i in range(1, 10):
    train_path = f"hackathon_data/stock_{i}_train.csv"
    results_df = compare_core_models_cv(train_path, cv=5)
    core_model_results[i] = results_df

    print(f"\nStock {i}")
    print(results_df)


Stock 1
              model  CV_RMSE_mean  CV_RMSE_std
0  LinearRegression      4.867444     0.030132
1             Ridge      4.867444     0.030134
2  GradientBoosting      6.855602     0.075542

Stock 2
              model  CV_RMSE_mean  CV_RMSE_std
0  LinearRegression      9.694009     0.448590
1             Ridge      9.694097     0.446819
2  GradientBoosting     22.467344     0.347374

Stock 3
              model  CV_RMSE_mean  CV_RMSE_std
1             Ridge     25.922219     9.432560
0  LinearRegression     26.315374     8.250030
2  GradientBoosting     44.275689    14.680465

Stock 4
              model  CV_RMSE_mean  CV_RMSE_std
2  GradientBoosting     24.753129     0.484115
1             Ridge     25.786832     0.472872
0  LinearRegression     25.786845     0.472883

Stock 5
              model  CV_RMSE_mean  CV_RMSE_std
1             Ridge     27.968289     3.925456
0  LinearRegression     27.977056     3.926786
2  GradientBoosting     29.374586     3.484674

Stock 6
      

In [34]:
core_model_summary_rows = []

for i in range(1, 10):
    results_df = core_model_results[i]
    best_row = results_df.iloc[0]

    core_model_summary_rows.append({
        "stock": i,
        "best_model": best_row["model"],
        "best_CV_RMSE": round(best_row["CV_RMSE_mean"], 4),
        "best_CV_RMSE_std": round(best_row["CV_RMSE_std"], 4)
    })

core_model_summary_df = pd.DataFrame(core_model_summary_rows)
core_model_summary_df

,stock,best_model,best_CV_RMSE,best_CV_RMSE_std
0,1,LinearRegression,4.8674,0.0301
1,2,LinearRegression,9.6940,0.4486
2,3,Ridge,25.9222,9.4326
3,4,GradientBoosting,24.7531,0.4841
4,5,Ridge,27.9683,3.9255
5,6,Ridge,56.0904,14.3739
6,7,GradientBoosting,15.5221,0.2780
7,8,Ridge,26.1218,0.5956
8,9,GradientBoosting,52.1810,12.5773


In [35]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import pandas as pd

def compare_boosting_models_cv(train_csv_path, cv=5):
    df = pd.read_csv(train_csv_path)

    X = df.drop(columns=["target"])
    y = df["target"]

    models = {}

    from xgboost import XGBRegressor
    models["XGBoost"] = Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("model", XGBRegressor(
            n_estimators=100,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            objective="reg:squarederror",
            verbosity=0
        ))
    ])

    from lightgbm import LGBMRegressor
    models["LightGBM"] = Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("model", LGBMRegressor(
            n_estimators=100,
            learning_rate=0.05,
            max_depth=-1,
            random_state=42,
            verbose=-1
        ))
    ])

    rows = []

    for model_name, model in models.items():
        rmse_scores = -cross_val_score(
            model,
            X,
            y,
            cv=cv,
            scoring="neg_root_mean_squared_error",
            n_jobs=-1
        )

        rows.append({
            "model": model_name,
            "CV_RMSE_mean": rmse_scores.mean(),
            "CV_RMSE_std": rmse_scores.std()
        })

    return pd.DataFrame(rows).sort_values("CV_RMSE_mean")

In [36]:
compare_boosting_models_cv("hackathon_data/stock_9_train.csv")

,model,CV_RMSE_mean,CV_RMSE_std
1,LightGBM,48.150754,13.448415
0,XGBoost,52.461279,12.235617


In [37]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
import pandas as pd

def compare_all_strong_models_cv(train_csv_path, cv=5):
    df = pd.read_csv(train_csv_path)

    X = df.drop(columns=["target"])
    y = df["target"]

    models = {
        "LinearRegression": Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler", StandardScaler()),
            ("model", LinearRegression())
        ]),

        "Ridge": Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=1.0))
        ]),

        "GradientBoosting": Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("model", GradientBoostingRegressor(
                n_estimators=100,
                learning_rate=0.05,
                max_depth=3,
                random_state=42
            ))
        ])
    }

    from xgboost import XGBRegressor
    models["XGBoost"] = Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("model", XGBRegressor(
            n_estimators=100,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            objective="reg:squarederror",
            verbosity=0
        ))
    ])

    from lightgbm import LGBMRegressor
    models["LightGBM"] = Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("model", LGBMRegressor(
            n_estimators=100,
            learning_rate=0.05,
            max_depth=-1,
            random_state=42,
            verbose=-1
        ))
    ])

    rows = []

    for model_name, model in models.items():
        rmse_scores = -cross_val_score(
            model,
            X,
            y,
            cv=cv,
            scoring="neg_root_mean_squared_error",
            n_jobs=-1
        )

        rows.append({
            "model": model_name,
            "CV_RMSE_mean": rmse_scores.mean(),
            "CV_RMSE_std": rmse_scores.std()
        })

    return pd.DataFrame(rows).sort_values("CV_RMSE_mean")

In [38]:
all_strong_results = {}

for i in range(1, 10):
    train_path = f"hackathon_data/stock_{i}_train.csv"
    results_df = compare_all_strong_models_cv(train_path, cv=5)
    all_strong_results[i] = results_df

    print(f"\nStock {i}")
    print(results_df)


Stock 1
              model  CV_RMSE_mean  CV_RMSE_std
0  LinearRegression      4.867444     0.030132
1             Ridge      4.867444     0.030134
4          LightGBM      5.518587     0.011029
3           XGBoost      5.827942     0.040886
2  GradientBoosting      6.855602     0.075542

Stock 2
              model  CV_RMSE_mean  CV_RMSE_std
0  LinearRegression      9.694009     0.448590
1             Ridge      9.694097     0.446819
4          LightGBM     18.282724     0.415526
3           XGBoost     19.870650     0.444785
2  GradientBoosting     22.467344     0.347374

Stock 3
              model  CV_RMSE_mean  CV_RMSE_std
1             Ridge     25.922219     9.432560
0  LinearRegression     26.315374     8.250030
3           XGBoost     39.157552    14.426171
2  GradientBoosting     44.275689    14.680465
4          LightGBM     79.355433    16.740304

Stock 4
              model  CV_RMSE_mean  CV_RMSE_std
3           XGBoost     24.588584     0.558616
4          LightGBM     

In [39]:
all_strong_summary_rows = []

for i in range(1, 10):
    results_df = all_strong_results[i]
    best_row = results_df.iloc[0]

    all_strong_summary_rows.append({
        "stock": i,
        "best_model": best_row["model"],
        "best_CV_RMSE": round(best_row["CV_RMSE_mean"], 4),
        "best_CV_RMSE_std": round(best_row["CV_RMSE_std"], 4)
    })

all_strong_summary_df = pd.DataFrame(all_strong_summary_rows)
all_strong_summary_df

,stock,best_model,best_CV_RMSE,best_CV_RMSE_std
0,1,LinearRegression,4.8674,0.0301
1,2,LinearRegression,9.6940,0.4486
2,3,Ridge,25.9222,9.4326
3,4,XGBoost,24.5886,0.5586
4,5,Ridge,27.9683,3.9255
5,6,Ridge,56.0904,14.3739
6,7,XGBoost,15.4754,0.2828
7,8,Ridge,26.1218,0.5956
8,9,LightGBM,48.1508,13.4484


In [40]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

def get_model_v2(model_name):
    if model_name == "LinearRegression":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler", StandardScaler()),
            ("model", LinearRegression())
        ])
    
    elif model_name == "Ridge":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=1.0))
        ])
    
    elif model_name == "GradientBoosting":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("model", GradientBoostingRegressor(
                n_estimators=100,
                learning_rate=0.05,
                max_depth=3,
                random_state=42
            ))
        ])
    
    elif model_name == "XGBoost":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("model", XGBRegressor(
                n_estimators=100,
                learning_rate=0.05,
                max_depth=4,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                objective="reg:squarederror",
                verbosity=0
            ))
        ])
    
    elif model_name == "LightGBM":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("model", LGBMRegressor(
                n_estimators=100,
                learning_rate=0.05,
                max_depth=-1,
                random_state=42,
                verbose=-1
            ))
        ])
    
    else:
        raise ValueError(f"Unknown model name: {model_name}")

In [42]:
from sklearn.model_selection import cross_val_score
import pandas as pd
import numpy as np

def get_cv_rmse_v2(train_csv_path, model_name, cv=5):
    df = pd.read_csv(train_csv_path)

    X = df.drop(columns=["target"])
    y = df["target"]

    model = get_model_v2(model_name)

    rmse_scores = -cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    )

    return rmse_scores.mean(), rmse_scores.std()

In [43]:
def fit_model_and_predict_v2(train_csv_path, test_csv_path, model_name):
    train_df = pd.read_csv(train_csv_path)
    test_df = pd.read_csv(test_csv_path)

    X_train = train_df.drop(columns=["target"])
    y_train = train_df["target"]

    model = get_model_v2(model_name)
    model.fit(X_train, y_train)

    predictions = model.predict(test_df)
    return predictions

In [44]:
def make_quotes(predicted_fair_value, cv_rmse, k=1.0):
    half_spread = k * cv_rmse
    bid = round(predicted_fair_value - half_spread, 2)
    ask = round(predicted_fair_value + half_spread, 2)
    spread_width = round(ask - bid, 2)
    return bid, ask, spread_width

In [45]:
all_quotes_v2 = {}
summary_rows_v2 = []

for i in range(1, 10):
    train_path = f"hackathon_data/stock_{i}_train.csv"
    test_path = f"hackathon_data/stock_{i}_test.csv"

    model_name = best_model_map_v2[i]

    # CV RMSE = uncertainty
    cv_rmse_mean, cv_rmse_std = get_cv_rmse_v2(train_path, model_name, cv=5)

    # Predict fair value on the test row
    fair_values = fit_model_and_predict_v2(train_path, test_path, model_name)

    k = k_map[i]

    stock_quotes = []

    for j, fair_value in enumerate(fair_values):
        bid, ask, spread = make_quotes(fair_value, cv_rmse_mean, k=k)

        stock_quotes.append({
            "row": j,
            "model_used": model_name,
            "predicted_fair_value": round(fair_value, 4),
            "cv_rmse_uncertainty": round(cv_rmse_mean, 4),
            "cv_rmse_std": round(cv_rmse_std, 4),
            "k": k,
            "bid": bid,
            "ask": ask,
            "spread_width": spread
        })

    stock_quotes_df = pd.DataFrame(stock_quotes)
    all_quotes_v2[i] = stock_quotes_df

    first_row = stock_quotes_df.iloc[0]
    summary_rows_v2.append({
        "stock": i,
        "model_used": first_row["model_used"],
        "predicted_fair_value": first_row["predicted_fair_value"],
        "cv_rmse_uncertainty": first_row["cv_rmse_uncertainty"],
        "cv_rmse_std": first_row["cv_rmse_std"],
        "k": first_row["k"],
        "bid": first_row["bid"],
        "ask": first_row["ask"],
        "spread_width": first_row["spread_width"]
    })

summary_v2_df = pd.DataFrame(summary_rows_v2)
summary_v2_df

c:\Users\rkade\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


,stock,model_used,predicted_fair_value,cv_rmse_uncertainty,cv_rmse_std,k,bid,ask,spread_width
0,1,LinearRegression,273.929300,4.8674,0.0301,1.00,269.06,278.80,9.74
1,2,LinearRegression,220.878200,9.6940,0.4486,1.00,211.18,230.57,19.39
2,3,Ridge,257.823700,25.9222,9.4326,1.25,225.42,290.23,64.81
3,4,XGBoost,238.279404,24.5886,0.5586,1.50,201.40,275.16,73.76
4,5,Ridge,249.028100,27.9683,3.9255,1.50,207.08,290.98,83.90
5,6,Ridge,174.920200,56.0904,14.3739,2.00,62.74,287.10,224.36
6,7,XGBoost,214.300995,15.4754,0.2828,1.75,187.22,241.38,54.16
7,8,Ridge,200.597900,26.1218,0.5956,2.00,148.35,252.84,104.49
8,9,LightGBM,219.513400,48.1508,13.4484,2.50,99.14,339.89,240.75


In [46]:
def classify_stock(row):
    rmse = row["cv_rmse_uncertainty"]
    rmse_std = row["cv_rmse_std"]

    if rmse < 10:
        risk_label = "Low"
        action = "Aggressive candidate"
    elif rmse < 20:
        risk_label = "Medium"
        action = "Moderate"
    elif rmse < 35:
        risk_label = "High"
        action = "Defensive"
    else:
        risk_label = "Very High"
        action = "Avoid market making"

    if rmse_std > 8:
        action = action + " (unstable CV)"

    return pd.Series([risk_label, action])

summary_v2_df[["risk_label", "action_recommendation"]] = summary_v2_df.apply(
    classify_stock, axis=1
)

summary_v2_df

,stock,model_used,predicted_fair_value,cv_rmse_uncertainty,cv_rmse_std,k,bid,ask,spread_width,risk_label,action_recommendation
0,1,LinearRegression,273.929300,4.8674,0.0301,1.00,269.06,278.80,9.74,Low,Aggressive candidate
1,2,LinearRegression,220.878200,9.6940,0.4486,1.00,211.18,230.57,19.39,Low,Aggressive candidate
2,3,Ridge,257.823700,25.9222,9.4326,1.25,225.42,290.23,64.81,High,Defensive (unstable CV)
3,4,XGBoost,238.279404,24.5886,0.5586,1.50,201.40,275.16,73.76,High,Defensive
4,5,Ridge,249.028100,27.9683,3.9255,1.50,207.08,290.98,83.90,High,Defensive
5,6,Ridge,174.920200,56.0904,14.3739,2.00,62.74,287.10,224.36,Very High,Avoid market making (unstable CV)
6,7,XGBoost,214.300995,15.4754,0.2828,1.75,187.22,241.38,54.16,Medium,Moderate
7,8,Ridge,200.597900,26.1218,0.5956,2.00,148.35,252.84,104.49,High,Defensive
8,9,LightGBM,219.513400,48.1508,13.4484,2.50,99.14,339.89,240.75,Very High,Avoid market making (unstable CV)


In [47]:
comparison_v2_df = summary_cv_df.merge(
    summary_v2_df,
    on="stock",
    suffixes=("_old", "_new")
)

comparison_v2_df[[
    "stock",
    "model_used_old",
    "model_used_new",
    "predicted_fair_value_old",
    "predicted_fair_value_new",
    "cv_rmse_uncertainty_old",
    "cv_rmse_uncertainty_new",
    "spread_width_old",
    "spread_width_new"
]]

,stock,model_used_old,model_used_new,predicted_fair_value_old,predicted_fair_value_new,cv_rmse_uncertainty_old,cv_rmse_uncertainty_new,spread_width_old,spread_width_new
0,1,Ridge,LinearRegression,273.9279,273.929300,4.8674,4.8674,9.74,9.74
1,2,Ridge,LinearRegression,220.9115,220.878200,9.6941,9.6940,19.39,19.39
2,3,Ridge,Ridge,257.8237,257.823700,25.9222,25.9222,64.81,64.81
3,4,GradientBoosting,XGBoost,235.8573,238.279404,24.7461,24.5886,74.24,73.76
4,5,Ridge,Ridge,249.0281,249.028100,27.9683,27.9683,83.90,83.90
5,6,Ridge,Ridge,174.9202,174.920200,56.0904,56.0904,224.36,224.36
6,7,GradientBoosting,XGBoost,213.8173,214.300995,15.5156,15.4754,54.31,54.16
7,8,Ridge,Ridge,200.5979,200.597900,26.1218,26.1218,104.49,104.49
8,9,GradientBoosting,LightGBM,210.1559,219.513400,52.9891,48.1508,264.95,240.75
